# Tool

Tool mở rộng khả năng của các [agent](https://docs.langchain.com/oss/python/langchain/agents) - cho phép chúng lấy dữ liệu theo thời gian thực, thực thi code, truy vấn cơ sở dữ liệu bên ngoài và thực hiện các hành động trong thế giới thực.

Về bản chất, tool là các hàm có thể gọi với đầu vào và đầu ra được xác định rõ ràng, được truyền cho một [chat model](https://docs.langchain.com/oss/python/langchain/models). Model sẽ quyết định khi nào nên gọi một tool dựa trên ngữ cảnh hội thoại, cũng như xác định các tham số đầu vào cần thiết.

<div class="alert alert-success">

Để biết chi tiết về cách các model xử lý tool call, hãy xem [tool calling](https://docs.langchain.com/oss/python/langchain/models#tool-calling). Theo dõi tool call và gỡ lỗi bằng [LangSmith](https://smith.langchain.com?utm_source=docs\&utm_medium=cta\&utm_campaign=langsmith-signup\&utm_content=oss-langchain-tools). Làm theo hướng dẫn [tracing quickstart](https://docs.langchain.com/langsmith/trace-with-langchain) để thiết lập.

Chúng tôi cũng khuyên bạn nên thiết lập [LangSmith Engine](https://docs.langchain.com/langsmith/engine) để giám sát các trace, phát hiện sự cố và đề xuất các bản sửa lỗi.

</div>

## Tạo tool

### Khai báo tool cơ bản

Cách đơn giản nhất để tạo một tool là sử dụng decorator [`@tool`](https://reference.langchain.com/python/langchain-core/tools/convert/tool). Mặc định, docstring của hàm sẽ trở thành mô tả của tool, giúp model hiểu khi nào nên sử dụng nó:

In [1]:
from langchain.tools import tool

@tool
def search_database(query: str, limit: int = 10) -> str:
    """Tìm kiếm các bản ghi khớp với truy vấn trong cơ sở dữ liệu khách hàng.

    Args:
        query: Các từ khóa cần tìm kiếm
        limit: Số lượng kết quả tối đa trả về
    """
    return f"Đã tìm thấy {limit} kết quả cho '{query}'"

Gợi ý kiểu dữ liệu là **bắt buộc** vì chúng xác định input schema của tool. Docstring nên chứa thông tin đầy đủ và súc tích để giúp model hiểu được mục đích của tool.

<div class="alert alert-info">

**Sử dụng tool phía server:** Một số chat model có sẵn các built-in tool (tìm kiếm web, trình thông dịch code) được thực thi ở server-side. Xem chi tiết tại phần sử dụng tool phía server.

</div>

<div class="alert alert-warning">

Nên sử dụng định dạng `snake_case` cho tên của tool (ví dụ: `web_search` thay vì `Web Search`). Một số nhà cung cấp model có thể gặp lỗi hoặc từ chối các tên chứa khoảng trắng hoặc ký tự đặc biệt. Việc chỉ sử dụng các ký tự chữ cái, chữ số, dấu gạch dưới và dấu gạch ngang sẽ giúp cải thiện khả năng tương thích giữa các nhà cung cấp.

</div>

### Tùy chỉnh thuộc tính của tool

#### Tên tool tùy chỉnh

Mặc định, tên của tool được lấy từ tên hàm. Bạn có thể ghi đè nó khi cần một cái tên mang tính mô tả hơn:

In [2]:
@tool("web_search")  # Tên tùy chỉnh
def search(query: str) -> str:
    """Tìm kiếm thông tin trên web."""
    return f"Kết quả cho: {query}"

print(search.name)

web_search


#### Mô tả tool tùy chỉnh

Ghi đè mô tả tự động tạo của tool để hướng dẫn model rõ ràng hơn:

In [5]:
@tool("calculator", description="Thực hiện các phép tính số học. Sử dụng cho bất kỳ bài toán nào.")
def calc(expression: str) -> str:
    """Đánh giá các biểu thức toán học."""
    return str(eval(expression))

print(calc.description)

Thực hiện các phép tính số học. Sử dụng cho bất kỳ bài toán nào.


### Khai báo schema nâng cao

Định nghĩa các đầu vào phức tạp bằng Pydantic model hoặc JSON schema:

In [6]:
# Pydantic model

from pydantic import BaseModel, Field
from typing import Literal

class WeatherInput(BaseModel):
    """Đầu vào cho các truy vấn thời tiết."""
    location: str = Field(description="Tên thành phố hoặc tọa độ")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="Tùy chọn đơn vị nhiệt độ"
    )
    include_forecast: bool = Field(
        default=False,
        description="Bao gồm dự báo trong 5 ngày tới"
    )

@tool(args_schema=WeatherInput)
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """Lấy thông tin thời tiết hiện tại và dự báo (tùy chọn)."""
    temp = 22 if units == "celsius" else 72
    result = f"Thời tiết hiện tại ở {location}: {temp} độ {units[0].upper()}"
    if include_forecast:
        result += "\n5 ngày tới: Trời nắng"
    return result

In [7]:
# JSON schema

weather_schema = {
    "type": "object",
    "properties": {
        "location": {"type": "string"},
        "units": {"type": "string"},
        "include_forecast": {"type": "boolean"}
    },
    "required": ["location", "units", "include_forecast"]
}

@tool(args_schema=weather_schema)
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """Lấy thông tin thời tiết hiện tại và dự báo (tùy chọn)."""
    temp = 22 if units == "celsius" else 72
    result = f"Thời tiết hiện tại ở {location}: {temp} độ {units[0].upper()}"
    if include_forecast:
        result += "\n5 ngày tới: Trời nắng"
    return result

### Các tên tham số dành riêng

Các tên tham số sau được hệ thống dành riêng và không thể dùng làm đối số cho tool. Việc sử dụng những tên này sẽ gây ra lỗi runtime.

| Tên tham số | Mục đích                                                                                         |
| ----------- | ------------------------------------------------------------------------------------------------ |
| `config`    | Dành riêng để truyền `RunnableConfig` nội bộ vào các tool                                        |
| `runtime`   | Dành riêng cho tham số `ToolRuntime` (dùng để truy cập vào state, context, store)                |

Để truy cập thông tin runtime, hãy sử dụng tham số [`ToolRuntime`](https://reference.langchain.com/python/langchain/tools/#langchain.tools.ToolRuntime) thay vì tự đặt tên tham số là `config` hoặc `runtime`.

## Truy cập context

Tool phát huy sức mạnh tối đa khi chúng có thể truy cập các thông tin runtime như lịch sử hội thoại, dữ liệu người dùng và bộ nhớ liên tục. Phần này trình bày cách truy cập và cập nhật những thông tin đó từ bên trong tool của bạn.

Tool có thể truy cập thông tin runtime thông qua tham số [`ToolRuntime`](https://reference.langchain.com/python/langchain/tools/#langchain.tools.ToolRuntime), bao gồm:

| Thành phần         | Mô tả                                                                                                                                           | Trường hợp sử dụng                                                                   |
| ------------------ | ----------------------------------------------------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------ |
| **State**          | Bộ nhớ ngắn hạn - dữ liệu có thể thay đổi, tồn tại cho cuộc hội thoại hiện tại (tin nhắn, bộ đếm, các trường tùy chỉnh)                         | Truy cập lịch sử hội thoại, theo dõi số lần tool call                                |
| **Context**        | Cấu hình bất biến được truyền vào tại thời điểm gọi (ID người dùng, thông tin session)                                                          |  Cá nhân hóa câu trả lời dựa trên danh tính người dùng                                |
| **Store**          | Bộ nhớ dài hạn - dữ liệu liên tục được giữ nguyên giữa các cuộc hội thoại khác nhau                                                             | Lưu tùy chọn người dùng, duy trì cơ sở tri thức                                      |
| **Stream Writer**  | Phát ra các bản cập nhật theo thời gian thực trong quá trình tool thực thi                                                                      | Hiển thị tiến trình cho các tác vụ chạy lâu                                          |
| **Execution Info** | Thông tin danh tính và trạng thái retry cho lần thực thi hiện tại (thread ID, run ID, số lần thử)                                               | Truy cập thread/run ID, điều chỉnh hành vi dựa trên trạng thái retry                 |
| **Server Info**    | Metadata của server khi chạy trên LangGraph Server (assistant ID, graph ID, người dùng đã xác thực)                                             | Truy cập thông tin của assistant ID, graph ID hoặc người dùng đã xác thực            |
| **Config**         | Các cấu hình [`RunnableConfig`](https://reference.langchain.com/python/langchain-core/runnables/config/RunnableConfig) cho quá trình thực thi   | Truy cập callback, tag và metadata                                                   |
| **Tool Call ID**   | Mã định danh duy nhất cho lần gọi tool hiện tại                                                                                                 | Tương quan các tool call để phục vụ cho việc lưu log và các lệnh gọi model           |

```mermaid theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
graph LR
    %% Runtime Context
    subgraph "🔧 Ngữ cảnh Tool Runtime"
        A[Tool Call] --> B[ToolRuntime]
        B --> C[Truy cập state]
        B --> D[Truy cập context]
        B --> E[Truy cập store]
        B --> F[Stream writer]
    end

    %% Available Resources
    subgraph "📊 Tài nguyên khả dụng"
        C --> G[Tin nhắn]
        C --> H[State tùy chỉnh]
        D --> I[ID người dùng]
        D --> J[Thông tin session]
        E --> K[Bộ nhớ dài hạn]
        E --> L[Tùy chọn người dùng]
    end

    %% Tool Capabilities
    subgraph "⚡ Khả năng nâng cao của Tool"
        M[Tool nhận biết ngữ cảnh]
        N[Tool có trạng thái]
        O[Tool hỗ trợ bộ nhớ]
        P[Tool streaming]
    end

    %% Connections
    G --> M
    H --> N
    I --> M
    J --> M
    K --> O
    L --> O
    F --> P

    classDef trigger fill:#F6FFDB,stroke:#6E8900,stroke-width:2px,color:#2E3900
    classDef process fill:#E5F4FF,stroke:#006DDD,stroke-width:2px,color:#030710
    classDef output fill:#EBD0F0,stroke:#885270,stroke-width:2px,color:#441E33
    classDef neutral fill:#F2FAFF,stroke:#40668D,stroke-width:2px,color:#2F4B68

    class A trigger
    class B,C,D,E,F process
    class G,H,I,J,K,L neutral
    class M,N,O,P output
```

### Bộ nhớ ngắn hạn (State)

State đại diện cho bộ nhớ ngắn hạn, tồn tại trong suốt quá trình diễn ra một cuộc hội thoại. Nó bao gồm lịch sử tin nhắn và bất kỳ trường tùy chỉnh nào mà bạn định nghĩa trong [graph state](https://docs.langchain.com/oss/python/langgraph/graph-api#state).

#### Truy cập state

Thêm `runtime: ToolRuntime` vào signature của tool để truy cập state. Vào thời điểm gọi, [`ToolNode`](https://reference.langchain.com/python/langgraph/agents/#langgraph.prebuilt.tool_node.ToolNode) sẽ tự động tiêm giá trị này vào; tham số này không được bao gồm trong schema của tool gửi cho model. Sử dụng `runtime.state` để đọc trạng thái cuộc hội thoại hiện tại:

In [ ]:
from langchain.tools import tool, ToolRuntime
from langchain.messages import HumanMessage

@tool
def get_last_user_message(runtime: ToolRuntime) -> str:
    """Lấy tin nhắn gần đây nhất từ người dùng."""
    messages = runtime.state["messages"]

    # Tìm tin nhắn cuối cùng của người dùng
    for message in reversed(messages):
        if isinstance(message, HumanMessage):
            return message.content

    return "Không tìm thấy tin nhắn nào của người dùng"

# Truy cập các trường state tùy chỉnh
@tool
def get_user_preference(
    pref_name: str,
    runtime: ToolRuntime
) -> str:
    """Lấy giá trị tùy chọn của người dùng."""
    preferences = runtime.state.get("user_preferences", {})
    return preferences.get(pref_name, "Chưa được thiết lập")

<div class="alert alert-warning">

Tham số `runtime` được ẩn khỏi model. Đối với ví dụ ở trên, model chỉ nhìn thấy `pref_name` trong tool schema.

</div>

#### Cập nhật state

Sử dụng [`Command`](https://reference.langchain.com/python/langgraph/types/Command) để cập nhật state của agent. Tính năng này hữu ích cho các tool cần cập nhật các trường state tùy chỉnh.
Hãy bao gồm một `ToolMessage` trong lệnh cập nhật để model có thể thấy được kết quả của tool call:

In [10]:
from langchain.agents import AgentState
from langchain.messages import ToolMessage
from langchain.tools import ToolRuntime, tool
from langgraph.types import Command


class CustomState(AgentState):
    user_name: str


@tool
def set_user_name(new_name: str, runtime: ToolRuntime[None, CustomState]) -> Command:
    """Thiết lập tên của người dùng trong trạng thái hội thoại."""
    return Command(
        update={
            "user_name": new_name,
            "messages": [
                ToolMessage(
                    content=f"Tên người dùng đã được thiết lập thành {new_name}.",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )

<div class="alert alert-success">

Khi tool cập nhật các biến state, hãy cân nhắc định nghĩa một [reducer](https://docs.langchain.com/oss/python/langgraph/graph-api#reducers) cho các trường đó. Vì các LLM có thể gọi song song nhiều tool, reducer sẽ xác định cách giải quyết xung đột khi một trường state bị cập nhật đồng thời bởi nhiều tool call.

</div>

### Context

Context cung cấp dữ liệu cấu hình bất biến được truyền vào tại thời điểm gọi. Hãy sử dụng context cho các ID người dùng, chi tiết session hoặc các cài đặt riêng biệt của ứng dụng mà không được thay đổi trong suốt cuộc hội thoại.

<div class="alert alert-info">

Trong khi `thread_id` (được truyền thông qua `config={"configurable": {"thread_id": ...}}`) giới hạn phạm vi của *cuộc hội thoại*: lịch sử tin nhắn và checkpoint, thì `context` mang theo dữ liệu *của từng phiên chạy* mà tool và middleware của bạn đọc vào thời điểm gọi. Trong môi trường production, bạn thường truyền cả hai tham số này cùng lúc: một `thread_id` cố định cho mỗi cuộc hội thoại và một đối tượng `context` trên mỗi lần invoke.

</div>

Truy cập context thông qua `runtime.context`. Hãy truyền nó kèm với một `thread_id` để cuộc hội thoại có thể được lưu trữ xuyên suốt các lượt trao đổi:

In [21]:
from dataclasses import dataclass

from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langchain_core.utils.uuid import uuid7


USER_DATABASE = {
    "user123": {
        "name": "Alice Johnson",
        "account_type": "Premium",
        "balance": 5000,
        "email": "alice@example.com",
    },
    "user456": {
        "name": "Bob Smith",
        "account_type": "Standard",
        "balance": 1200,
        "email": "bob@example.com",
    },
}


@dataclass
class UserContext:
    user_id: str


@tool
def get_account_info(runtime: ToolRuntime[UserContext]) -> str:
    """Lấy thông tin tài khoản của người dùng hiện tại."""
    user_id = runtime.context.user_id

    if user_id in USER_DATABASE:
        user = USER_DATABASE[user_id]
        return (
            f"Chủ tài khoản: {user['name']}\n"
            f"Loại tài khoản: {user['account_type']}\n"
            f"Số dư: ${user['balance']}"
        )
    return "Không tìm thấy người dùng"


agent = create_agent(
    "google_genai:gemini-3.5-flash-lite",
    tools=[get_account_info],
    context_schema=UserContext,
    system_prompt="Bạn là một trợ lý tài chính.",
)

agent.invoke(
    {"messages": [{"role": "user", "content": "Số dư hiện tại của tôi là bao nhiêu?"}]},
    config={"configurable": {"thread_id": str(uuid7())}},
    context=UserContext(user_id="user123"),
)

{'messages': [HumanMessage(content='Số dư hiện tại của tôi là bao nhiêu?', additional_kwargs={}, response_metadata={}, id='6cb4ac09-20ab-45f2-9d59-80ed8668b01c'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_account_info', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'call_16849': 'El4KXAERTTIPaCU7QyMLoNJhRqlw57cW5YA8kg0Q7PL6xQBh84PeDRwNaFrHlIj6XsFqCBng6O1+nFzS4sHVuWFieDqFxpR/cdalIUN+o2q7njIHZgocQDudgCSTtJ51'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0478c-2376-7803-9651-ea97557c6f20-0', tool_calls=[{'name': 'get_account_info', 'args': {}, 'id': 'call_16849', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 46, 'output_tokens': 12, 'total_tokens': 58, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content='Chủ tài khoản: Alice Johnson\nLoại tài khoản: Premium\nSố dư: $5000', name='

### Bộ nhớ dài hạn (Store)

[`BaseStore`](https://reference.langchain.com/python/langchain-core/stores/BaseStore) cung cấp không gian lưu trữ liên tục được giữ nguyên giữa các cuộc hội thoại khác nhau. Không giống như state (bộ nhớ ngắn hạn), dữ liệu được lưu vào store sẽ vẫn có sẵn trong các session tương lai.

Truy cập store thông qua `runtime.store`. Store sử dụng mô hình namespace/key để tổ chức dữ liệu:

<div class="alert alert-success">

Đối với các hệ thống production, hãy sử dụng một implementation của persistent store như [`PostgresStore`](https://reference.langchain.com/python/langgraph/store/#langgraph.store.postgres.PostgresStore), `MongoDBStore`, hoặc `RedisStore` thay vì `InMemoryStore`. Xem tài liệu [memory](https://docs.langchain.com/oss/python/langgraph/add-memory) để biết chi tiết cách thiết lập.

In [20]:
from typing import Any
from langgraph.store.memory import InMemoryStore
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime

# Truy cập memory
@tool
def get_user_info(user_id: str, runtime: ToolRuntime) -> str:
    """Tra cứu thông tin người dùng."""
    store = runtime.store
    user_info = store.get(("users",), user_id)
    return str(user_info.value) if user_info else "Người dùng không xác định"

# Cập nhật memory
@tool
def save_user_info(user_id: str, user_info: dict[str, Any], runtime: ToolRuntime) -> str:
    """Lưu thông tin người dùng."""
    store = runtime.store
    store.put(("users",), user_id, user_info)
    return "Đã lưu thông tin người dùng thành công."

store = InMemoryStore()
agent = create_agent(
    "google_genai:gemini-3.5-flash-lite",
    tools=[get_user_info, save_user_info],
    store=store
)

# Session đầu tiên: lưu thông tin người dùng
agent.invoke({
    "messages": [{"role": "user", "content": "Lưu người dùng sau: userid: abc123, name: Foo, age: 25, email: foo@langchain.dev"}]
})

# Session thứ hai: lấy thông tin người dùng
agent.invoke({
    "messages": [{"role": "user", "content": "Lấy thông tin người dùng có id 'abc123'"}]
})

{'messages': [HumanMessage(content="Lấy thông tin người dùng có id 'abc123'", additional_kwargs={}, response_metadata={}, id='028da5e1-0485-4667-bc8d-e7d902cb3b67'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_user_info', 'arguments': '{"user_id": "abc123"}'}, '__gemini_function_call_thought_signatures__': {'call_17691': 'El4KXAERTTIPNGRXoAelwJJROdCPKvd5p7GbaFsqxTkqw1kqBlgFzOOn2zWgc0mU7LRPO9FGtsOWBN1hxHuwwfeqQ6/vmXZ1WijG/QgJTe7FpfFQojVyv4J8crV98QIl'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0478a-f3aa-7e43-8b71-c03949de0bca-0', tool_calls=[{'name': 'get_user_info', 'args': {'user_id': 'abc123'}, 'id': 'call_17691', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 131, 'output_tokens': 23, 'total_tokens': 154, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content="{'name': 'Foo', 'email': 'foo@langchain

### Stream writer

Stream các bản cập nhật theo thời gian thực từ các tool trong quá trình thực thi. Điều này hữu ích để cung cấp phản hồi về tiến trình cho người dùng trong các thao tác chạy mất nhiều thời gian.

Sử dụng `runtime.stream_writer` để phát ra các cập nhật tùy chỉnh:

In [ ]:
from langchain.tools import tool, ToolRuntime

@tool
def get_weather(city: str, runtime: ToolRuntime) -> str:
    """Lấy thông tin thời tiết cho một thành phố nhất định."""
    writer = runtime.stream_writer

    # Stream các bản cập nhật tùy chỉnh trong quá trình tool thực thi
    writer(f"Đang tìm kiếm dữ liệu cho thành phố: {city}")
    writer(f"Đã thu thập dữ liệu cho thành phố: {city}")

    return f"Trời luôn nắng ở {city}!"

<div class="alert alert-info">

Nếu bạn sử dụng `runtime.stream_writer` bên trong tool của mình, tool đó phải được gọi trong một ngữ cảnh thực thi của LangGraph. Xem [Streaming](https://docs.langchain.com/oss/python/langchain/streaming) để biết thêm chi tiết.

</div>

### Execution info

Truy cập thread ID, run ID, và trạng thái thử lại từ bên trong một tool thông qua `runtime.execution_info`:

In [22]:
from langchain.tools import tool, ToolRuntime

@tool
def log_execution_context(runtime: ToolRuntime) -> str:
    """Log thông tin định danh của quá trình thực thi."""
    info = runtime.execution_info
    print(f"Thread: {info.thread_id}, Run: {info.run_id}")
    print(f"Lần thử thứ: {info.node_attempt}")
    return "done"

<div class="alert alert-info">

Yêu cầu `deepagents>=0.5.0` (hoặc `langgraph>=1.1.5`).

</div>

### Server info

Khi tool của bạn chạy trên LangGraph Server, bạn có thể truy cập assistant ID, graph ID và người dùng đã xác thực thông qua `runtime.server_info`:

In [ ]:
from langchain.tools import tool, ToolRuntime

@tool
def get_assistant_scoped_data(runtime: ToolRuntime) -> str:
    """Lấy dữ liệu thuộc phạm vi của assistant hiện tại."""
    server = runtime.server_info
    if server is not None:
        print(f"Assistant: {server.assistant_id}, Graph: {server.graph_id}")
        if server.user is not None:
            print(f"User: {server.user.identity}")
    return "done"

## Thực thi tool

Trong LangChain, tool được sử dụng bởi các agent (ví dụ thông qua [`create_agent`](https://reference.langchain.com/python/langchain/agents/factory/create_agent)) và việc xử lý lỗi cho tool được cấu hình thông qua [middleware](/oss/python/langchain/middleware).

Đối với các workflow trên LangGraph, việc thực thi tool được đảm nhận bởi [`ToolNode`](https://reference.langchain.com/python/langgraph/agents/#langgraph.prebuilt.tool_node.ToolNode). Xem [ToolNode](https://docs.langchain.com/oss/python/langgraph/workflows-agents#toolnode) để biết cách sử dụng Graph API, bao gồm cách các tool có thể truy cập state hiện tại của graph cũng như run-scoped context.

### Các giá trị trả về của tool

Bạn có thể chọn các kiểu giá trị trả về khác nhau cho tool của mình:

* Trả về một `string` để thu được kết quả dễ đọc đối với người dùng.
* Trả về một `object` để có kết quả có cấu trúc mà model có thể phân tích cú pháp.
* Trả về một `Command` đi kèm với tin nhắn (tùy chọn) khi bạn cần ghi đè vào state.

#### Trả về một chuỗi

Hãy trả về chuỗi khi tool cần cung cấp plain text để model có thể đọc và sử dụng trong câu phản hồi tiếp theo.

In [ ]:
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    """Lấy thời tiết cho một thành phố."""
    return f"Hiện tại trời đang nắng ở {city}."

Hành vi:

* Giá trị trả về được chuyển đổi thành `ToolMessage`.
* Model nhìn thấy đoạn văn bản đó và quyết định sẽ làm gì tiếp theo.
* Các trường state của agent không bị thay đổi, trừ khi model hoặc một tool khác thực hiện việc này sau đó.

Hãy sử dụng phương pháp này khi kết quả là dạng văn bản dễ đọc.

#### Trả về một đối tượng

Hãy trả về một đối tượng (ví dụ: `dict`) khi tool của bạn tạo ra dữ liệu có cấu trúc mà model cần xem xét.

In [ ]:
from langchain.tools import tool

@tool
def get_weather_data(city: str) -> dict:
    """Lấy dữ liệu thời tiết có cấu trúc cho một thành phố."""
    return {
        "city": city,
        "temperature_c": 22,
        "conditions": "sunny",
    }

Hành vi:

* Đối tượng được serialize (tuần tự hóa) và gửi lại dưới dạng kết quả đầu ra của tool.
* Model có thể đọc các trường cụ thể và suy luận (reason) dựa trên đó.
* Tương tự như trả về chuỗi, việc này không cập nhật trực tiếp graph state.

Sử dụng cách này khi quá trình suy luận phía sau được hưởng lợi từ các trường dữ liệu rõ ràng thay vì văn bản tự do.

#### Trả về nội dung đa phương thức

Tool không bị giới hạn ở định dạng plain text. Khi model hỗ trợ kết quả đầu ra là đa phương thức, tool có thể trả về các [content block tiêu chuẩn](https://docs.langchain.com/oss/python/langchain/messages#standard-content-blocks) để model có thể nhận được cả văn bản, hình ảnh và các phương tiện truyền thông khác trong một kết quả trả về duy nhất.

In [ ]:
from langchain.tools import tool

@tool
def capture_screenshot() -> list[dict]:
    """Chụp ảnh màn hình của trang hiện tại."""
    return [
        {"type": "text", "text": "Ảnh chụp màn hình của trang hiện tại:"},
        {"type": "image", "url": "https://example.com/page.png"},
    ]

Hành vi:

* Giá trị trả về được chuyển đổi thành một `ToolMessage` với phần `content` là đa phương thức.
* Sử dụng `message.content_blocks` để đọc danh sách block đã được chuẩn hóa sau khi tool chạy xong.
* Model phải hỗ trợ các modality mà bạn trả về. Hãy kiểm tra [khả năng của model](https://docs.langchain.com/oss/python/integrations/chat) trước khi trả về hình ảnh, âm thanh hoặc video.

Về các loại block cũng như các yêu cầu cụ thể đối với từng nhà cung cấp, hãy xem [multimodal message](https://docs.langchain.com/oss/python/langchain/messages#multimodal). Các MCP tool có tính năng trả về hình ảnh hoặc nội dung hỗn hợp cũng được xử lý theo cách tương tự; xem [nội dung đa phương thức cho tool](https://docs.langchain.com/oss/python/langchain/mcp#multimodal-tool-content).

#### Trả về một Command

Hãy trả về một [`Command`](https://reference.langchain.com/python/langgraph/types/Command) khi tool cần cập nhật graph state (ví dụ: cài đặt tùy chọn người dùng hoặc trạng thái ứng dụng).
Khi `Command` trỏ tới graph hiện tại, hãy đính kèm một `ToolMessage` vào bản cập nhật có chứa tool call ID khớp với tool call hiện tại.
Mọi tool call trong lịch sử tin nhắn đều phải có một `ToolMessage` tương ứng.

Sử dụng `runtime.tool_call_id` cho tham số `tool_call_id`. `ToolNode` sẽ ép buộc thi hành quy định này: nếu lệnh cập nhật không chứa `ToolMessage` khớp với tool call, nó sẽ phát sinh lỗi `ValueError`.


In [ ]:
from langchain.messages import ToolMessage
from langchain.tools import ToolRuntime, tool
from langgraph.types import Command

@tool
def set_language(language: str, runtime: ToolRuntime) -> Command:
    """Thiết lập ngôn ngữ phản hồi ưa thích."""
    return Command(
        update={
            "preferred_language": language,
            "messages": [
                ToolMessage(
                    content=f"Ngôn ngữ đã được thiết lập thành {language}.",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )

Hành vi:

* Command cập nhật state thông qua khóa `update`.
* Phần state đã cập nhật sẽ khả dụng cho các bước tiếp theo trong cùng một lượt chạy.
* Hãy dùng các reducer cho những trường dữ liệu có khả năng bị cập nhật từ nhiều tool call song song.

Sử dụng cách này khi tool không chỉ cần trả về dữ liệu, mà còn trực tiếp thay đổi agent state.

#### Trả về trực tiếp từ tool

Cài đặt return direct trên một tool để rút ngắn vòng lặp agent: agent sẽ ngay lập tức trả kết quả của tool cho caller mà không cần đưa nó quay lại model để xử lý thêm.

In [23]:
from langchain.agents import create_agent
from langchain.tools import tool


@tool(return_direct=True)
def fetch_order_status(order_id: str) -> str:
    """Lấy trạng thái hiện tại của đơn hàng khách hàng."""
    # Trong môi trường production, hãy truy vấn hệ thống quản lý đơn hàng của bạn tại đây
    return f"Đơn hàng {order_id} đã được giao và sẽ đến nơi trong 2 ngày tới."


agent = create_agent(
    model="google_genai:gemini-3.5-flash",
    tools=[fetch_order_status],
)

agent.invoke({
    "messages": [{"role": "user", "content": "Trạng thái của đơn hàng #12345 là gì?"}]
})

{'messages': [HumanMessage(content='Trạng thái của đơn hàng #12345 là gì?', additional_kwargs={}, response_metadata={}, id='1b64f761-5ee5-421e-a28a-92d81b06170c'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'fetch_order_status', 'arguments': '{"order_id": "12345"}'}, '__gemini_function_call_thought_signatures__': {'call_26886': 'EpwFCpkFARFNMg8Sm6uJJDdbQbK6mpilief9alHkBmGaDxnVNrOW96wD1WACvMAHEQFPBuqOKRplBXQEOxyeWzajjgZRoPSFnsXEZrP7x0PWMhX3sALu7geKecmKP/5+LG5gjDaJ50lUTu1zeummOOrQ95wRWbFRgmCaserLsseymijzVxUH6ErgNTD2XC+jk4hJiugSGzm5/PMAGon7pl0QLlNjrSGLbdTpU4Kg2Y9iLk49MNzm0h+0ErluImSYmScBt9VI3GArb3PoBa/C2Utydvfsx2e3SnAL6JOWNKKpf7ce9dibWIuf6WxQu6+XBMxfWpc+lJt0xjE34RS/iU36V8qulOo6IUKTq6JJRoDOy5QEj4OAR0gez8gH3wSoq5Eb4hLithym0Ii3UdECBVLdyeP9rjVq6reg2lpReiNphAOh9To1ABE0VAs5N6L2QxUw+QQsJIi/WIPYAgibXfdwiZXW9lwwJvEq9n5cbO9wycnzi/LmVNaojQcB9GldsLi5pLf9Qi2ez0/wu4MwDEmwZHUbrRkhi1+yvGwpcVLH3ATX3XamnrV3xv+UdeDcQm2NgdCtPIZBk5CvfDAn+QprWCuT/IpUXAa9XNaG73rVoGjUT6FIcHOpYTMRn1YxnAZ

Hành vi:

* Tool thực thi bình thường và output của nó được bao bọc trong một `ToolMessage`.
* Agent kết thúc vòng lặp và trả về output của tool như câu phản hồi cuối cùng, bỏ qua mọi quá trình model call tiếp theo.
* **Các lệnh gọi tool song song:** Khi model gọi đồng thời nhiều tool trong một bước, tất cả chúng sẽ được thực thi trước. Sau khi tất cả các tool kết thúc, agent chỉ trỏ đến trạng thái `END` nếu **mọi** tool trong batch đều có `return_direct=True`. Phản hồi cuối cùng sẽ bao gồm output (dưới dạng `ToolMessage`) của tất cả các tool được gọi trong bước đó.

Sử dụng tính năng này khi:

* Output của tool là một câu trả lời hoàn chỉnh, có thể sử dụng ngay (ví dụ: tra cứu và trả về kết quả sẵn sàng để hiển thị).
* Bạn muốn tránh việc gọi thêm model không cần thiết khi không đòi hỏi suy luận logic thêm.
* Bạn cần dữ liệu đầu ra mang tính định định, không bị sửa đổi: model không thể viết lại, tóm tắt hoặc thực hiện hành động trên kết quả của tool.

<div class="alert alert-warning">

Bởi vì model không trực tiếp xử lý kết quả của tool, `return_direct=True` không phù hợp cho những tool mà kết quả đầu ra của nó đòi hỏi phải có quá trình suy luận, tóm tắt hoặc kết nối thêm cùng các tool khác.

</div>

<div class="alert alert-warning">

**Các cuộc gọi song song hỗn hợp:** Nếu model gọi một tool có `return_direct=True` đồng thời với các tool không có `return_direct=True`, agent sẽ **không** thoát khỏi luồng ở bước đó. Nó sẽ đưa mọi `ToolMessage` từ batch này quay ngược về model, để model có thể đánh giá tất cả các kết quả. Tính năng `return_direct` chỉ cắt ngắn vòng lặp khi mọi tool call trong bước đó đều được đặt `return_direct=True`.

</div>

#### Trả về một Command đi kèm return_direct

Một tool với tùy chọn `return_direct=True` cũng có thể trả về một [`Command`](https://reference.langchain.com/python/langgraph/types/Command) để cập nhật graph state trước khi agent kết thúc. Không giống như các giá trị trả về thông thường, một `Command` không tự động bị biến thành `ToolMessage`. Khi `Command` nhắm mục tiêu vào graph hiện tại (`graph` không được thiết lập hoặc bằng `None`), hãy thêm một `ToolMessage` vào `Command.update` khớp với `tool_call_id` của tool. Bỏ qua bước này sẽ khiến `ToolNode` báo lỗi `ValueError`, bởi vì mọi tool call `AIMessage` đều bắt buộc phải có một `ToolMessage` tương ứng trong lịch sử tin nhắn.

In [ ]:
from langchain.messages import ToolMessage
from langchain.tools import ToolRuntime, tool
from langgraph.types import Command


@tool(return_direct=True)
def fetch_and_store_order(order_id: str, runtime: ToolRuntime) -> Command:
    """Lấy trạng thái đơn hàng và lưu nó vào state."""
    status = f"Đơn hàng {order_id} đã được giao và sẽ đến nơi trong 2 ngày tới."
    return Command(
        update={
            "last_order_status": status,
            # Bắt buộc đính kèm ToolMessage để lịch sử tin nhắn hợp lệ
            "messages": [
                ToolMessage(
                    content=status,
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )

Thay vào đó, nếu bạn cần ghi đè lên graph cấp cha, hãy đặt `graph=Command.PARENT`. Trong trường hợp này, yêu cầu bắt buộc có `ToolMessage` bị loại bỏ vì quá trình thực thi đã thoát ra ngoài graph hiện tại.

### Xử lý lỗi

Xử lý các lỗi của tool bằng [middleware](https://docs.langchain.com/oss/python/langchain/middleware) từ LangChain agent để tiến hành retry khi tool bị lỗi hoặc trả về các thông báo lỗi tùy chỉnh:

In [25]:
from collections.abc import Callable

from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage
from langchain.tools.tool_node import ToolCallRequest


@wrap_tool_call
def handle_tool_errors(
    request: ToolCallRequest,
    handler: Callable[[ToolCallRequest], ToolMessage],
) -> ToolMessage:
    """Chuyển đổi các ngoại lệ của tool thành các ToolMessage mà model có thể xử lý."""
    try:
        return handler(request)
    except Exception as e:
        return ToolMessage(
            content=f"Lỗi tool: Vui lòng kiểm tra lại đầu vào và thử lại. ({e})",
            tool_call_id=request.tool_call["id"],
        )


agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[],
    middleware=[handle_tool_errors],
)

### State injection

Tool truy cập vào graph state thông qua [`ToolRuntime`](https://reference.langchain.com/python/langchain/tools/#langchain.tools.ToolRuntime). Xem mục [truy cập context](https://docs.langchain.com/oss/python/langchain/tools#access-context) để biết thêm thông tin về các API liên quan đến state, context, store và streaming.

In [ ]:
from langchain.tools import tool, ToolRuntime

@tool
def get_message_count(runtime: ToolRuntime) -> str:
    """Lấy số lượng tin nhắn trong cuộc hội thoại."""
    messages = runtime.state["messages"]
    return f"Có {len(messages)} tin nhắn."

Để biết thêm chi tiết về cách truy cập state, context và bộ nhớ dài hạn từ các tools, hãy xem [truy cập context](https://docs.langchain.com/oss/python/langchain/tools#access-context).

## Lựa chọn tool động

Với các tool động, tập hợp những tool có sẵn cho agent sẽ được thay đổi vào lúc runtime thay vì phải được định nghĩa toàn bộ từ đầu. Không phải tool nào cũng phù hợp với mọi trường hợp. Quá nhiều tool có thể làm model bị quá tải và gia tăng lỗi sai; trong khi quá ít tool thì lại hạn chế khả năng của model. Việc chọn lựa tool động cho phép điều chỉnh kho công cụ sẵn có dựa trên trạng thái xác thực, quyền người dùng, feature flag, hoặc giai đoạn của cuộc hội thoại.

Có hai phương pháp tùy thuộc vào việc các tool đã được định trước hay chưa:

<Tabs>
  <Tab title="Lọc các tools đã được đăng ký trước (pre-registered)">
    Khi mọi tools khả thi đều đã được biết trước vào lúc khởi tạo agent, bạn có thể đăng ký trước tất cả chúng và sử dụng tính năng lọc động để chọn ra những tools nào sẽ được tiếp cận với model dựa trên trạng thái (state), quyền hạn hoặc context.

    <Tabs>
      <Tab title="State">
        Chỉ kích hoạt các tools cấp độ cao sau khi trải qua một số mốc quan trọng trong cuộc hội thoại:

        ```python theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
        from langchain.agents import create_agent
        from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
        from typing import Callable

        @wrap_model_call
        def state_based_tools(
            request: ModelRequest,
            handler: Callable[[ModelRequest], ModelResponse]
        ) -> ModelResponse:
            """Lọc các tools dựa trên State của cuộc hội thoại."""
            # Đọc từ State: kiểm tra xem người dùng đã xác thực chưa
            state = request.state
            is_authenticated = state.get("authenticated", False)
            message_count = len(state["messages"])

            # Chỉ kích hoạt các tools nhạy cảm sau khi đã xác thực
            if not is_authenticated:
                tools = [t for t in request.tools if t.name.startswith("public_")]
                request = request.override(tools=tools)
            elif message_count < 5:
                # Giới hạn các tools ở đầu cuộc hội thoại
                tools = [t for t in request.tools if t.name != "advanced_search"]
                request = request.override(tools=tools)

            return handler(request)

        agent = create_agent(
            model="gpt-5.5",
            tools=[public_search, private_search, advanced_search],
            middleware=[state_based_tools]
        )
        ```
      </Tab>

      <Tab title="Store">
        Lọc tools dựa trên tùy chọn của người dùng hoặc feature flags trong Store:

        ```python theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
        from dataclasses import dataclass
        from langchain.agents import create_agent
        from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
        from typing import Callable
        from langgraph.store.memory import InMemoryStore

        @dataclass
        class Context:
            user_id: str

        @wrap_model_call
        def store_based_tools(
            request: ModelRequest,
            handler: Callable[[ModelRequest], ModelResponse]
        ) -> ModelResponse:
            """Lọc tools dựa trên các thiết lập trong Store."""
            user_id = request.runtime.context.user_id

            # Đọc từ Store: lấy các tính năng (features) được bật cho người dùng
            store = request.runtime.store
            feature_flags = store.get(("features",), user_id)

            if feature_flags:
                enabled_features = feature_flags.value.get("enabled_tools", [])
                # Chỉ bao gồm những tools được bật cho người dùng này
                tools = [t for t in request.tools if t.name in enabled_features]
                request = request.override(tools=tools)

            return handler(request)

        agent = create_agent(
            model="gpt-5.5",
            tools=[search_tool, analysis_tool, export_tool],
            middleware=[store_based_tools],
            context_schema=Context,
            store=InMemoryStore()
        )
        ```
      </Tab>

      <Tab title="Runtime Context">
        Lọc tools dựa trên quyền của người dùng từ Runtime Context:

        ```python theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
        from dataclasses import dataclass
        from langchain.agents import create_agent
        from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
        from typing import Callable

        @dataclass
        class Context:
            user_role: str

        @wrap_model_call
        def context_based_tools(
            request: ModelRequest,
            handler: Callable[[ModelRequest], ModelResponse]
        ) -> ModelResponse:
            """Lọc tools dựa trên quyền từ Runtime Context."""
            # Đọc từ Runtime Context: lấy vai trò (role) của người dùng
            if request.runtime is None or request.runtime.context is None:
                # Nếu không có context được cung cấp, mặc định là viewer (hạn chế nhất)
                user_role = "viewer"
            else:
                user_role = request.runtime.context.user_role

            if user_role == "admin":
                # Admin được dùng tất cả các tools
                pass
            elif user_role == "editor":
                # Editor không thể xóa
                tools = [t for t in request.tools if t.name != "delete_data"]
                request = request.override(tools=tools)
            else:
                # Viewer chỉ được dùng các tools đọc (read-only)
                tools = [t for t in request.tools if t.name.startswith("read_")]
                request = request.override(tools=tools)

            return handler(request)

        agent = create_agent(
            model="gpt-5.5",
            tools=[read_data, write_data, delete_data],
            middleware=[context_based_tools],
            context_schema=Context
        )
        ```
      </Tab>
    </Tabs>

    Cách tiếp cận này rất tốt khi:

    * Tất cả các tools có thể dùng được đều đã biết trước vào thời điểm compile/startup
    * Bạn muốn phân loại dựa trên quyền hạn (permissions), feature flags hoặc trạng thái cuộc trò chuyện
    * Bản thân các tools là tĩnh (static) nhưng trạng thái sẵn sàng sử dụng (availability) của chúng là động

    Xem [Cách chọn tool động (Dynamically selecting tools)](/oss/python/langchain/middleware/custom#dynamically-selecting-tools) để có thêm ví dụ.
  </Tab>

  <Tab title="Đăng ký tool trong quá trình runtime">
    Khi tools được khám phá hoặc khởi tạo vào lúc runtime (ví dụ: load từ MCP server, generate từ dữ liệu người dùng, hoặc lấy từ remote registry), bạn cần phải đăng ký đồng thời xử lý quá trình thực thi của chúng một cách động.

    Cách này yêu cầu có hai middleware hooks:

    1. `wrap_model_call` - Thêm các tools động vào request
    2. `wrap_tool_call` - Xử lý quá trình thực thi của các tools được thêm động này

    ```python theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
    from langchain.tools import tool
    from langchain.agents import create_agent
    from langchain.agents.middleware import AgentMiddleware, ModelRequest, ToolCallRequest

    # Một tool sẽ được thêm động vào lúc runtime
    @tool
    def calculate_tip(bill_amount: float, tip_percentage: float = 20.0) -> str:
        """Tính tiền tip (tiền boa) cho hóa đơn."""
        tip = bill_amount * (tip_percentage / 100)
        return f"Tiền tip: ${tip:.2f}, Tổng cộng: ${bill_amount + tip:.2f}"

    class DynamicToolMiddleware(AgentMiddleware):
        """Middleware đăng ký và xử lý các tools động."""

        def wrap_model_call(self, request: ModelRequest, handler):
            # Thêm tool động vào request
            # Phần này có thể được load từ một MCP server, cơ sở dữ liệu, v.v.
            updated = request.override(tools=[*request.tools, calculate_tip])
            return handler(updated)

        def wrap_tool_call(self, request: ToolCallRequest, handler):
            # Xử lý quá trình thực thi cho tool động
            if request.tool_call["name"] == "calculate_tip":
                return handler(request.override(tool=calculate_tip))
            return handler(request)

    agent = create_agent(
        model="gpt-5.5",
        tools=[get_weather],  # Chỉ các tools tĩnh (static) mới đăng ký ở đây
        middleware=[DynamicToolMiddleware()],
    )

    # Agent giờ đây có thể sử dụng cả get_weather VÀ calculate_tip
    result = agent.invoke({
        "messages": [{"role": "user", "content": "Tính khoản tip 20% cho hóa đơn $85"}]
    })
    ```

    Phương pháp này hiệu quả khi:

    * Tools được khám phá trong runtime (ví dụ: từ MCP server)
    * Tools được sinh ra một cách linh hoạt dựa vào dữ liệu hoặc cấu hình từ người dùng
    * Bạn đang tích hợp hệ thống với một external tool registries

    <Note>
      Hook `wrap_tool_call` là bắt buộc đối với các tools đăng ký vào thời điểm runtime vì agent cần biết cách thực thi các tools vốn không có trong danh sách gốc. Thiếu đi hook này, agent sẽ không thể tìm được cách gọi (invoke) các tool mới được thêm vào.
    </Note>
  </Tab>
</Tabs>

## Headless tools

Một số tools cần phải được chạy **tại vị trí ứng dụng của người dùng đang hoạt động** (thường là trên trình duyệt), thay vì thực thi trực tiếp trên server. **Headless tools** (tools không có thành phần thực thi đi kèm) là tập hợp các định nghĩa (definitions) của một tool, bao gồm tên, mô tả và argument schema. Các định nghĩa này sẽ được bạn đăng ký trên **server** để dùng cho agent của mình. **Thực thi (implementation)** của chúng sẽ chỉ được ghi nhận ở **client** và hoạt động sau một quá trình ngắt quãng/tiếp tục (interrupt/resume handshake) ngắn.

Điều này khác với các tools thông thường mà phần hàm chính (function body) sẽ được chạy trên server, và cũng khác với [sử dụng tool phía server (server-side tool use)](#server-side-tool-use) - nơi nhà cung cấp model thực thi các built-in tools từ xa.

### Khi nào nên sử dụng headless tools

Hãy dùng chúng khi công việc của bạn phụ thuộc vào **môi trường, thiết bị, hoặc giao diện UI** vốn chỉ có mặt ở client. Ví dụ:

* **Các API của trình duyệt:** Geolocation (Định vị), IndexedDB, Clipboard (Bảng nhớ tạm), Canvas 2D, File pickers, Battery API, v.v.
* **Quyền riêng tư và cục bộ (locality):** Dữ liệu được lưu trữ nguyên vẹn trên thiết bị (ví dụ: các vùng "bộ nhớ" cục bộ trong IndexedDB).
* **Độ trễ:** Không có khoảng thời gian gián đoạn do phải gửi yêu cầu lên server cho các thao tác diễn ra hoàn toàn cục bộ.
* **Hiệu ứng an toàn, có cấu trúc:** Nên ưu tiên tạo ra nhiều tools nhỏ và được định kiểu cụ thể (ví dụ: mỗi hàm nguyên thủy canvas là một tool riêng) thay vì truyền code tùy ý cho hàm `eval`.

### Cơ chế hoạt động của mô hình (pattern) này

Ở cả hai runtimes, model vẫn thấy một tool thông thường mà nó có quyền được gọi, tuy nhiên quá trình xử lý thật sự diễn ra bên ngoài tiến trình server.

1. **Định nghĩa (Define)** một headless tool bằng hàm `tool(name=..., description=..., args_schema=...)` từ `langchain.tools`. Headless tool hoàn toàn chỉ bao gồm schema, không có khối implementation nằm trong tiến trình (in-process) server.
2. **Đăng ký (Register)** tool đó vào hàm `create_agent` hoặc LangGraph graph của bạn, để model có thể gọi tới nó như bình thường.
3. **Xử lý (Handle)** payload bị gián đoạn (interrupt) khi tool được kích hoạt. Thay vì chạy ngay tại local, graph sẽ bị dừng (pause) kèm theo một chuỗi payload dạng `{"type": "tool", "tool_call": {"id", "name", "args"}}`.
4. **Tiếp tục (Resume)** quá trình xử lý graph sau khi ứng dụng của bạn, hay một hệ thống khác, hoặc từ quy trình đánh giá con người vừa thực thi xong hành động tương ứng. Đối với các luồng hoạt động chạy trên trình duyệt (browser-based), bạn có thể tiến hành sao chép (mirror) schema tương tự bên frontend và đính kèm lệnh `.implement(...)` ở đó.

<Info>
  Nếu bạn gọi `tool(...)` trong Python nhưng chỉ có các tham số `name`, `description` và `args_schema`, LangChain sẽ mặc định trả về một đối tượng `HeadlessTool`. Hiện tại không có API `.implement()` phía Python.
</Info>

Khi model phát lệnh gọi tới một trong những tools này, run (chuyến chạy thực thi) sẽ bị **interrupt** thay vì thực thi ngay tool tại cục bộ. Ứng dụng của bạn sẽ kiểm tra payload của tool, tiến hành kích hoạt chức năng ở đúng môi trường thực thi (chẳng hạn như là một trình duyệt, một dịch vụ nền tảng khác hay khâu xem xét từ quản trị viên), sau đó cho phép graph tiếp tục **resume** với kết quả tool nhận được. Với các hỗ trợ qua JS SDK hooks, hệ thống có khả năng nhận diện các ngắt (interrupts) từ headless-tool, tiến hành gọi client implementation phù hợp và gửi lại một câu lệnh resume thay cho bạn.

Đừng quên sử dụng callback **`onTool`** (tùy chọn) để quan sát các vòng đời sự kiện (lifecycle events) (`start`, `success`, `error`) để cung cấp phản hồi trên UI, chẳng hạn như cho các biểu tượng xoay chờ (spinners) hoặc thông báo nảy (toasts).

<Card title="Mô hình frontend cho headless tools" href="/oss/python/langchain/frontend/headless-tools" icon="device-desktop" arrow="true" horizontal>
  Xem ví dụ end-to-end về cách các schema-only tools thực thi bên phía client cùng hàm `useStream`.
</Card>

## Các tools có sẵn (Prebuilt tools)

LangChain cung cấp một thư viện quy mô về các prebuilt tools và toolkits dành cho đa dạng nhiệm vụ thường gặp như tìm kiếm trên website, thông dịch mã lập trình (code interpretation), kết nối với cơ sở dữ liệu (database access) và nhiều thứ khác. Các tools có sẵn này có thể tích hợp thẳng trực tiếp cho agents của bạn mà không yêu cầu tự tay viết bất kỳ đoạn code tùy chỉnh nào.

Bạn có thể xem thêm mục tích hợp (integration) tại phần [tools and toolkits](/oss/python/integrations/tools) để biết danh sách chi tiết các tính năng cụ thể tùy theo chuyên mục cần thiết.

## Sử dụng tool phía server

Một vài chat models được tích hợp sẵn chức năng gọi các built-in tools chạy phía server side thuộc quản lý của nhà cung cấp model (provider). Tính năng này sở hữu khả năng đặc biệt như truy xuất web search hoặc thông dịch mã (code interpreters) mà không đòi hỏi bạn phải tốn công viết các phần logic để host trực tiếp những công cụ kể trên.

Hãy tham khảo cụ thể phần tài liệu tích hợp trên trang [chat model integration pages](/oss/python/integrations/providers) cũng như [tool calling documentation](/oss/python/langchain/models#server-side-tool-use) nếu bạn mong muốn có thông tin rõ hơn về thao tác mở cũng như sử dụng tính năng built-in tools này.

***

<div className="source-links">
  <Callout icon="terminal-2">
    [Kết nối tài liệu này](/use-these-docs) vào Claude, VSCode, và nhiều hơn thế thông qua MCP để được hỗ trợ câu trả lời trong thời gian thực.
  </Callout>

  <Callout icon="edit">
    [Chỉnh sửa trang này trên GitHub](https://github.com/langchain-ai/docs/edit/main/src/oss/langchain/tools.mdx) hoặc [tạo báo cáo sự cố (issue)](https://github.com/langchain-ai/docs/issues/new/choose).
  </Callout>
</div>